In [3]:
!pip install tensorflow scikit-learn imbalanced-learn tldextract shap matplotlib pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 2.7 MB/s eta 0:00:00a 0:00:01


In [ ]:
"""
Refactored end-to-end pipeline for phishing URL detection on the PhiUSIIL
dataset, reconstructed from:

"A Dynamic Phishing URL Detection System Integrating Advanced Machine
Learning and Crowdsourced Threat Intelligence" (Aldhuwayhi & Alromih, 2026)

DATASET
-------
Kaggle: https://www.kaggle.com/datasets/ndarvind/phiusiil-phishing-url-dataset
    kaggle datasets download -d ndarvind/phiusiil-phishing-url-dataset -p ./data --unzip

"""

import os
import re
import random
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.feature_selection import f_classif, chi2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, classification_report
)

from imblearn.over_sampling import SMOTE

import tldextract


# ===========================================================================
# 0. CONFIG & REPRODUCIBILITY
# ===========================================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Works for both local runs and Kaggle notebooks; override as needed.
DATA_PATH = os.environ.get(
    "PHIUSIIL_CSV_PATH",
    "/kaggle/input/datasets/ndarvind/phiusiil-phishing-url-dataset/PhiUSIIL_Phishing_URL_Dataset.csv"
    if os.path.exists("/kaggle/input")
    else "./data/PhiUSIIL_Phishing_URL_Dataset.csv",
)

LABEL_COL = "label"
URL_COL = "URL"
TITLE_COL = "Title"
DOMAIN_COL = "Domain"
TLD_COL = "TLD"

# ---------------------------------------------------------------------
# LABEL MAPPING — READ THIS
# ---------------------------------------------------------------------
# PhiUSIIL's own Kaggle documentation and the source paper's reported
# Counter({1: 134850, 0: 100945}) both point to:
#     1 = legitimate, 0 = phishing
# This is the OPPOSITE of what was assumed in an earlier draft. Rather
# than hardcode either version silently, the loader below asserts the
# configured mapping against the actual value_counts() of your CSV and
# fails loudly if they don't match — check that assertion output the
# first time you run this against your actual file.
LABEL_LEGITIMATE = 1
LABEL_PHISHING = 0
TARGET_NAMES_IN_LABEL_ORDER = ["Phishing", "Legitimate"]  # index 0, index 1

# Column schema (from the actual PhiUSIIL CSV header)
TEXT_COLS = [URL_COL, TITLE_COL]

CATEGORICAL_BINARY_COLS = [
    "IsDomainIP", "HasObfuscation", "IsHTTPS", "HasTitle", "HasFavicon",
    "Robots", "IsResponsive", "HasDescription", "HasExternalFormSubmit",
    "HasSocialNet", "HasSubmitButton", "HasHiddenFields", "HasPasswordField",
    "Bank", "Pay", "Crypto", "HasCopyrightInfo",
]

CONTINUOUS_COLS = [
    "URLSimilarityIndex", "CharContinuationRate", "TLDLegitimateProb",
    "URLCharProb", "TLDLength", "NoOfSubDomain", "NoOfObfuscatedChar",
    "ObfuscationRatio", "NoOfLettersInURL", "LetterRatioInURL",
    "NoOfDegitsInURL", "DegitRatioInURL", "NoOfEqualsInURL",
    "NoOfQMarkInURL", "NoOfAmpersandInURL", "NoOfOtherSpecialCharsInURL",
    "SpacialCharRatioInURL", "LineOfCode", "LargestLineLength",
    "DomainTitleMatchScore", "URLTitleMatchScore", "NoOfURLRedirect",
    "NoOfSelfRedirect", "NoOfPopup", "NoOfiFrame", "NoOfImage", "NoOfCSS",
    "NoOfJS", "NoOfSelfRef", "NoOfEmptyRef", "NoOfExternalRef",
]

# Paper explicitly names these as manually-dropped noisy columns
MANUAL_NOISY_DROP = ["URLLength", "DomainLength"]

# Domain is near-unique per row (an identifier, not a generalizable
# feature) and is dropped. TLD is high-cardinality categorical; it's
# bucketed to the top-N most frequent values + "other" before one-hot
# encoding to avoid a huge sparse explosion.
IDENTIFIER_DROP = [DOMAIN_COL]
TOP_N_TLDS = 20

MAX_URL_LEN = 200          # not specified in paper
VOCAB_SIZE = 8000          # not specified in paper
EMBEDDING_DIM = 50         # stated in paper
BATCH_SIZE = 32            # not specified in paper
EPOCHS = 30                # upper bound; EarlyStopping will cut this short
LEARNING_RATE = 1e-3       # not specified in paper
TARGET_N_FEATURES = 34     # stated in paper as the final selected count

OUTPUT_DIR = "./artifacts"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ===========================================================================
# 1. LOAD DATA
# ===========================================================================
def load_data(path: str = DATA_PATH) -> pd.DataFrame:
    df = pd.read_csv(path)
    print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")

    counts = df[LABEL_COL].value_counts().to_dict()
    print(f"Raw label distribution: {counts}")

    # Sanity check the configured mapping against the actual majority/
    # minority split reported in the paper (134,850 legitimate vs
    # 100,945 phishing). This will not catch every possible mislabeling,
    # but it will catch the mapping being flipped outright.
    majority_label = max(counts, key=counts.get)
    if majority_label != LABEL_LEGITIMATE:
        raise AssertionError(
            f"Configured LABEL_LEGITIMATE={LABEL_LEGITIMATE} but the "
            f"majority class in the CSV is label={majority_label} "
            f"(counts={counts}). The paper reports legitimate URLs as "
            f"the majority class (134,850 vs 100,945 phishing) — double "
            f"check LABEL_LEGITIMATE/LABEL_PHISHING at the top of this file."
        )
    return df


# ===========================================================================
# 2. PREPROCESSING (Section IV.B.2)
# ===========================================================================
def clean_domain_text(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    def strip_www(url: str) -> str:
        if not isinstance(url, str):
            return url
        return re.sub(r"^https?://www\.", "https://", url, flags=re.IGNORECASE)

    def clean_text(text: str) -> str:
        if not isinstance(text, str):
            return ""
        return text.encode("ascii", errors="ignore").decode()

    if URL_COL in df.columns:
        df[URL_COL] = df[URL_COL].apply(strip_www).apply(clean_text)

    if TITLE_COL in df.columns:
        df[TITLE_COL] = df[TITLE_COL].apply(clean_text)

    # tldextract correctly handles multi-part TLDs (co.uk, gov.in, com.au)
    def get_registered_domain(url: str) -> str:
        if not isinstance(url, str) or not url:
            return ""
        ext = tldextract.extract(url)
        return ext.domain

    df["MainDomain"] = df[URL_COL].apply(get_registered_domain)
    return df


def enforce_data_integrity(df: pd.DataFrame) -> pd.DataFrame:
    before = len(df)
    df = df.dropna(subset=[LABEL_COL])
    df = df.drop_duplicates()
    print(f"Data integrity: dropped {before - len(df)} rows (dupes / missing label)")
    return df


def bucket_top_n_tld(df: pd.DataFrame, top_n: int = TOP_N_TLDS) -> pd.DataFrame:
    df = df.copy()
    if TLD_COL not in df.columns:
        return df
    top_values = df[TLD_COL].value_counts().nlargest(top_n).index
    df[TLD_COL] = df[TLD_COL].where(df[TLD_COL].isin(top_values), other="other_tld")
    return df


def one_hot_encode_categoricals(df: pd.DataFrame, categorical_cols: list) -> pd.DataFrame:
    existing = [c for c in categorical_cols if c in df.columns]
    if not existing:
        return df
    return pd.get_dummies(df, columns=existing, drop_first=True)


# ===========================================================================
# 3. FEATURE SELECTION — FIT ON TRAIN ONLY (fixes leakage issue #3, #5)
# ===========================================================================
def select_features_train_only(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    p_threshold: float = 0.05,
    target_n_features: int = TARGET_N_FEATURES,
):
    """
    Returns:
        selected_columns: list[str]
        fitted_scaler: MinMaxScaler fit on X_train (only used internally
                        for the Chi2 test on binary columns; the numeric
                        branch has its own separate StandardScaler later)
    """
    X_train = X_train.drop(
        columns=[c for c in MANUAL_NOISY_DROP if c in X_train.columns],
        errors="ignore",
    )
    X_numeric = X_train.select_dtypes(include=[np.number]).fillna(0)

    scaler = MinMaxScaler()
    X_scaled = pd.DataFrame(
        scaler.fit_transform(X_numeric),  # fit on TRAIN ONLY
        columns=X_numeric.columns, index=X_numeric.index,
    )

    is_binary = X_scaled.nunique() <= 2
    cat_cols = X_scaled.columns[is_binary]
    cont_cols = X_scaled.columns[~is_binary]

    anova_p = pd.Series(dtype=float)
    chi2_p = pd.Series(dtype=float)

    if len(cont_cols) > 0:
        _, p_vals = f_classif(X_scaled[cont_cols], y_train)
        anova_p = pd.Series(p_vals, index=cont_cols)

    if len(cat_cols) > 0:
        _, p_vals = chi2(X_scaled[cat_cols], y_train)
        chi2_p = pd.Series(p_vals, index=cat_cols)

    all_p = pd.concat([anova_p, chi2_p])
    selected = all_p[all_p < p_threshold].index.tolist()

    if len(selected) > target_n_features:
        selected = all_p.loc[selected].sort_values().head(target_n_features).index.tolist()

    print(f"Feature selection (train-only fit): {len(selected)} features retained")
    return selected, scaler


# ===========================================================================
# 4. CLASS BALANCING — SMOTE, with a text-preserving neighbor lookup
# ===========================================================================
def smote_with_text(X_num_train: pd.DataFrame, url_text_train: pd.Series, y_train: pd.Series):
    """
    SMOTE synthesizes new numeric feature vectors by interpolating between
    real minority-class neighbors. Those synthetic rows have no real URL
    string. Rather than silently duplicating or fabricating URLs, each
    synthetic row is assigned the URL text of its nearest REAL neighbor
    (found via NearestNeighbors over the original training features).
    This keeps the text branch's input distribution grounded in real
    URLs, at the cost of some synthetic/text pairs being approximate.

    NOTE: imbalanced-learn's SMOTE returns original samples first
    (unchanged, in their original order) followed by synthetic samples
    appended at the end. This function relies on that ordering — verify
    it holds for your installed imblearn version if you see mismatches.
    """
    sm = SMOTE(random_state=SEED)
    X_res, y_res = sm.fit_resample(X_num_train, y_train)

    n_original = len(X_num_train)
    n_synthetic = len(X_res) - n_original
    print(f"SMOTE: {n_original} original rows -> {len(X_res)} total "
          f"({n_synthetic} synthetic rows added)")

    if n_synthetic > 0:
        nn = NearestNeighbors(n_neighbors=1).fit(X_num_train.values)
        synthetic_rows = X_res.iloc[n_original:].values
        _, nn_idx = nn.kneighbors(synthetic_rows)
        synthetic_text = url_text_train.iloc[nn_idx.flatten()].reset_index(drop=True)
        text_res = pd.concat(
            [url_text_train.reset_index(drop=True), synthetic_text],
            ignore_index=True,
        )
    else:
        text_res = url_text_train.reset_index(drop=True)

    return X_res.reset_index(drop=True), text_res, pd.Series(y_res).reset_index(drop=True)


# ===========================================================================
# 5. TEXT TOKENIZATION — FIT ON TRAIN ONLY (fixes leakage issue #4)
# ===========================================================================
def build_text_input(df: pd.DataFrame) -> pd.Series:
    """Concatenates URL + Title into one string for the single text branch
    shown in Figure 3 (paper describes both as 'sequential text data' but
    the architecture diagram has only one text InputLayer)."""
    url = df[URL_COL].fillna("").astype(str) if URL_COL in df.columns else ""
    title = df[TITLE_COL].fillna("").astype(str) if TITLE_COL in df.columns else ""
    return (url + " [TITLE] " + title).astype(str)


def fit_tokenizer(train_texts: pd.Series) -> Tokenizer:
    tokenizer = Tokenizer(num_words=VOCAB_SIZE, char_level=True, oov_token="<OOV>")
    tokenizer.fit_on_texts(train_texts)  # TRAIN ONLY
    return tokenizer


def texts_to_padded(texts: pd.Series, tokenizer: Tokenizer) -> np.ndarray:
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_URL_LEN, padding="post", truncating="post")


# ===========================================================================
# 6. MODEL ARCHITECTURE (Section IV.B.3 / Figure 3, + regularization)
# ===========================================================================
def build_model(num_numerical_features: int, vocab_size: int = VOCAB_SIZE) -> Model:
    # --- Text branch ---
    text_input = layers.Input(shape=(MAX_URL_LEN,), name="url_text_input")
    x = layers.Embedding(vocab_size, EMBEDDING_DIM, name="embedding")(text_input)
    x = layers.Dropout(0.2, name="embedding_dropout")(x)

    lstm_seq = layers.LSTM(100, return_sequences=True, name="lstm_1")(x)
    lstm_seq = layers.Dropout(0.3, name="lstm1_dropout")(lstm_seq)

    # AdditiveAttention (Bahdanau-style) instead of dot-product Attention —
    # closer to the paper's description of weighting critical URL portions.
    attn_out = layers.AdditiveAttention(name="attention")([lstm_seq, lstm_seq])

    text_repr = layers.LSTM(64, name="lstm_2")(attn_out)
    text_repr = layers.Dropout(0.3, name="lstm2_dropout")(text_repr)

    # --- Numeric branch ---
    numeric_input = layers.Input(shape=(num_numerical_features,), name="numeric_features_input")
    numeric_repr = layers.Dense(64, activation="relu", name="numeric_dense")(numeric_input)
    numeric_repr = layers.Dropout(0.3, name="numeric_dropout")(numeric_repr)

    # --- Merge + classify ---
    merged = layers.Concatenate(name="concatenate")([text_repr, numeric_repr])
    merged = layers.BatchNormalization(name="batch_norm")(merged)
    output = layers.Dense(1, activation="sigmoid", name="output")(merged)

    model = Model(inputs=[text_input, numeric_input], outputs=output,
                  name="hybrid_lstm_attention_phishing_detector")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="binary_crossentropy",
        metrics=["accuracy",
                 tf.keras.metrics.Precision(name="precision"),
                 tf.keras.metrics.Recall(name="recall"),
                 tf.keras.metrics.AUC(name="auc")],
    )
    return model


# ===========================================================================
# 7. EVALUATION
# ===========================================================================
def evaluate(model, X_text_test, X_num_test, y_test, plot: bool = True):
    y_prob = model.predict([X_text_test, X_num_test]).ravel()
    y_pred = (y_prob >= 0.5).astype(int)

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob),
        "pr_auc": average_precision_score(y_test, y_prob),
    }
    for k, v in metrics.items():
        print(f"{k:>10}: {v:.4f}")

    print(classification_report(y_test, y_pred, target_names=TARGET_NAMES_IN_LABEL_ORDER))

    cm = confusion_matrix(y_test, y_pred)
    if plot:
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        ConfusionMatrixDisplay(cm, display_labels=TARGET_NAMES_IN_LABEL_ORDER).plot(
            ax=axes[0], cmap="Blues", colorbar=False
        )
        axes[0].set_title("Confusion Matrix")

        fpr, tpr, _ = roc_curve(y_test, y_prob)
        axes[1].plot(fpr, tpr, label=f"ROC AUC = {metrics['roc_auc']:.4f}")
        axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray")
        axes[1].set_xlabel("False Positive Rate")
        axes[1].set_ylabel("True Positive Rate")
        axes[1].set_title("ROC Curve")
        axes[1].legend()

        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, "evaluation_plots.png"), dpi=150)
        plt.close()
        print(f"Saved evaluation plots to {OUTPUT_DIR}/evaluation_plots.png")

    return metrics


def plot_learning_curves(history):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].plot(history.history["accuracy"], label="Train Accuracy")
    axes[0].plot(history.history["val_accuracy"], label="Validation Accuracy")
    axes[0].set_title("Accuracy over Epochs")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(history.history["loss"], label="Train Loss")
    axes[1].plot(history.history["val_loss"], label="Validation Loss")
    axes[1].set_title("Loss over Epochs")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "learning_curves.png"), dpi=150)
    plt.close()
    print(f"Saved learning curves to {OUTPUT_DIR}/learning_curves.png")


# ===========================================================================
# 8. EXPLAINABILITY (numeric branch only — see caveat)
# ===========================================================================
def explain_numeric_branch(model, X_num_background: np.ndarray, X_num_sample: np.ndarray,
                            feature_names: list, fixed_text_input: np.ndarray):
    """
    CAVEAT: SHAP support for multi-input (text + numeric) Keras models is
    not a clean, off-the-shelf story. This function explains sensitivity
    of the *numeric* branch only, holding the text input fixed at a
    constant background sequence (e.g. all-padding). Treat this as a
    partial, numeric-only explanation, not a full accounting of the
    model's behavior (which also depends heavily on the text branch).
    """
    try:
        import shap
    except ImportError:
        print("shap not installed (`pip install shap`) — skipping explainability step.")
        return None

    def predict_fn(x_num):
        n = x_num.shape[0]
        text_batch = np.repeat(fixed_text_input[:1], n, axis=0)
        return model.predict([text_batch, x_num]).ravel()

    explainer = shap.KernelExplainer(predict_fn, X_num_background[:50])
    shap_values = explainer.shap_values(X_num_sample[:20], nsamples=100)

    shap.summary_plot(shap_values, X_num_sample[:20], feature_names=feature_names, show=False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "shap_summary_numeric_branch.png"), dpi=150)
    plt.close()
    print(f"Saved SHAP summary to {OUTPUT_DIR}/shap_summary_numeric_branch.png")
    return shap_values


# ===========================================================================
# 9. INFERENCE HELPERS
# ===========================================================================
def predict_from_row(model, tokenizer, num_scaler, selected_features,
                      row: dict) -> dict:
    """
    Use this when you already have a full feature row (e.g. re-scoring a
    row from the dataset, or a row produced by your own feature-extraction
    pipeline matching the PhiUSIIL schema). This does NOT scrape a live
    URL — see extract_live_features() below for that (much harder) case.
    """
    text = str(row.get(URL_COL, "")) + " [TITLE] " + str(row.get(TITLE_COL, ""))
    X_text = texts_to_padded(pd.Series([text]), tokenizer)

    num_values = [row.get(f, 0) for f in selected_features]
    X_num = num_scaler.transform([num_values])

    prob = float(model.predict([X_text, X_num]).ravel()[0])
    pred_label = LABEL_LEGITIMATE if prob >= 0.5 else LABEL_PHISHING
    pred_name = "Legitimate" if pred_label == LABEL_LEGITIMATE else "Phishing"
    return {"probability_legitimate": prob, "predicted_label": pred_label,
            "predicted_class": pred_name}


def extract_live_features(url: str) -> dict:
    """
    STUB — NOT a full implementation.

    Roughly a third of PhiUSIIL's columns (LineOfCode, LargestLineLength,
    HasFavicon, Robots, IsResponsive, NoOfURLRedirect, NoOfSelfRedirect,
    HasDescription, NoOfPopup, NoOfiFrame, HasExternalFormSubmit,
    HasSocialNet, HasSubmitButton, HasHiddenFields, HasPasswordField,
    HasCopyrightInfo, NoOfImage, NoOfCSS, NoOfJS, NoOfSelfRef,
    NoOfEmptyRef, NoOfExternalRef, DomainTitleMatchScore,
    URLTitleMatchScore, HasTitle, Title) require actually fetching and
    parsing the live page (requests + BeautifulSoup, checking for a
    favicon, counting iframes/forms/scripts, resolving redirects, etc.).
    That's a genuinely separate engineering task — a working scraper with
    error handling, timeouts, and HTML parsing — not something that can
    be responsibly stubbed out in a few lines.

    What CAN be computed from the URL string alone (no page fetch) is
    implemented below as a starting point.
    """
    ext = tldextract.extract(url)
    features = {
        "IsHTTPS": int(url.startswith("https")),
        "NoOfSubDomain": ext.subdomain.count(".") + (1 if ext.subdomain else 0),
        "NoOfLettersInURL": sum(c.isalpha() for c in url),
        "NoOfDegitsInURL": sum(c.isdigit() for c in url),
        "NoOfEqualsInURL": url.count("="),
        "NoOfQMarkInURL": url.count("?"),
        "NoOfAmpersandInURL": url.count("&"),
        "TLDLength": len(ext.suffix),
        # ... remaining ~25 features require a live page fetch; not
        # implemented here. Raise clearly rather than silently zero-filling:
    }
    raise NotImplementedError(
        "Only URL-string-derived features are implemented (see `features` "
        "dict above for what's available). Page-content-dependent features "
        "require a real HTTP fetch + HTML parser and are not included. "
        "Use predict_from_row() with a fully-featured row instead, or "
        "extend this function with a requests/BeautifulSoup scraper."
    )


# ===========================================================================
# 10. MAIN PIPELINE
# ===========================================================================
def main():
    # --- Load & clean ---
    df = load_data(DATA_PATH)
    df = clean_domain_text(df)
    df = enforce_data_integrity(df)
    df = bucket_top_n_tld(df)

    y = df[LABEL_COL].astype(int).reset_index(drop=True)
    text_input_full = build_text_input(df).reset_index(drop=True)

    drop_cols = [LABEL_COL, URL_COL, TITLE_COL, DOMAIN_COL, "MainDomain"] + IDENTIFIER_DROP
    X_raw = df.drop(columns=[c for c in drop_cols if c in df.columns]).reset_index(drop=True)
    X_raw = one_hot_encode_categoricals(X_raw, [TLD_COL] + CATEGORICAL_BINARY_COLS)

    # --- Split FIRST, before any fitting (fixes leakage #3, #4, #5) ---
    X_train_raw, X_test_raw, text_train, text_test, y_train, y_test = train_test_split(
        X_raw, text_input_full, y,
        test_size=0.2, stratify=y, random_state=SEED,
    )

    # --- Feature selection fit on TRAIN only ---
    selected_features, _ = select_features_train_only(X_train_raw, y_train)
    X_train_sel = X_train_raw[selected_features].fillna(0).reset_index(drop=True)
    X_test_sel = X_test_raw[selected_features].fillna(0).reset_index(drop=True)
    text_train = text_train.reset_index(drop=True)
    text_test = text_test.reset_index(drop=True)
    y_train = y_train.reset_index(drop=True)
    y_test = y_test.reset_index(drop=True)

    # --- Numeric scaling, fit on TRAIN only ---
    num_scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(
        num_scaler.fit_transform(X_train_sel), columns=selected_features
    )
    X_test_scaled = pd.DataFrame(
        num_scaler.transform(X_test_sel), columns=selected_features
    )

    # --- Class balancing via SMOTE (train only — never touch test) ---
    X_train_bal, text_train_bal, y_train_bal = smote_with_text(
        X_train_scaled, text_train, y_train
    )

    # --- Tokenizer fit on TRAIN (post-balancing) text only ---
    tokenizer = fit_tokenizer(text_train_bal)
    X_text_train = texts_to_padded(text_train_bal, tokenizer)
    X_text_test = texts_to_padded(text_test, tokenizer)

    # --- Build & train ---
    model = build_model(num_numerical_features=X_train_bal.shape[1])
    model.summary()

    callbacks = [
        EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
    ]

    history = model.fit(
        [X_text_train, X_train_bal.values], y_train_bal.values,
        validation_split=0.2,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
    )

    plot_learning_curves(history)

    # --- Evaluate on the untouched, non-resampled test set ---
    metrics = evaluate(model, X_text_test, X_test_scaled.values, y_test.values)

    # --- Save artifacts ---
    model.save(os.path.join(OUTPUT_DIR, "hybrid_model.keras"))
    with open(os.path.join(OUTPUT_DIR, "tokenizer.pkl"), "wb") as f:
        pickle.dump(tokenizer, f)
    with open(os.path.join(OUTPUT_DIR, "num_scaler.pkl"), "wb") as f:
        pickle.dump(num_scaler, f)
    with open(os.path.join(OUTPUT_DIR, "selected_features.pkl"), "wb") as f:
        pickle.dump(selected_features, f)
    print(f"Saved model + tokenizer + scaler + feature list to {OUTPUT_DIR}/")

    # --- Optional: SHAP on numeric branch (requires `pip install shap`) ---
    run_shap = False
    if run_shap:
        fixed_text = X_text_test[:1]
        explain_numeric_branch(
            model,
            X_num_background=X_train_bal.values,
            X_num_sample=X_test_scaled.values,
            feature_names=selected_features,
            fixed_text_input=fixed_text,
        )

    return model, metrics


if __name__ == "__main__":
    main()

Loaded dataset: 235795 rows, 55 columns
Raw label distribution: {1: 134850, 0: 100945}
Data integrity: dropped 0 rows (dupes / missing label)
Feature selection (train-only fit): 31 features retained
SMOTE: 188636 original rows -> 215760 total (27124 synthetic rows added)


Model: "hybrid_lstm_attention_phishing_detector"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ url_text_input      │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 200, 50)   │    400,000 │ url_text_input[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_dropout   │ (None, 200, 50)   │          0 │ embedding[0][0]   │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 200, 100)  │     60,400 │ embedding_dropou… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm1_dropout       │ (None, 200, 100)  │          0 │ lstm_1[0][0]      │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, 200, 100)  │        100 │ lstm1_dropout[0]… │
│ (AdditiveAttention) │                   │            │ lstm1_dropout[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ numeric_features_i… │ (None, 31)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ (None, 64)        │     42,240 │ attention[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ numeric_dense       │ (None, 64)        │      2,048 │ numeric_features… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm2_dropout       │ (None, 64)        │          0 │ lstm_2[0][0]      │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ numeric_dropout     │ (None, 64)        │          0 │ numeric_dense[0]… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 128)       │          0 │ lstm2_dropout[0]… │
│ (Concatenate)       │                   │            │ numeric_dropout[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_norm          │ (None, 128)       │        512 │ concatenate[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 1)         │        129 │ batch_norm[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 505,429 (1.93 MB)

 Trainable params: 505,173 (1.93 MB)

 Non-trainable params: 256 (1.00 KB)

Epoch 1/30
5394/5394 ━━━━━━━━━━━━━━━━━━━━ 334s 61ms/step - accuracy: 0.9916 - auc: 0.9995 - loss: 0.0243 - precision: 0.9914 - recall: 0.9940 - val_accuracy: 0.9958 - val_auc: 1.0000 - val_loss: 0.0110 - val_precision: 0.9806 - val_recall: 1.0000 - learning_rate: 0.0010
Epoch 2/30
5394/5394 ━━━━━━━━━━━━━━━━━━━━ 329s 61ms/step - accuracy: 0.9973 - auc: 0.9998 - loss: 0.0082 - precision: 0.9969 - recall: 0.9983 - val_accuracy: 0.9965 - val_auc: 1.0000 - val_loss: 0.0066 - val_precision: 0.9838 - val_recall: 1.0000 - learning_rate: 0.0010
Epoch 3/30
5394/5394 ━━━━━━━━━━━━━━━━━━━━ 330s 61ms/step - accuracy: 0.9978 - auc: 0.9998 - loss: 0.0068 - precision: 0.9976 - recall: 0.9986 - val_accuracy: 0.9978 - val_auc: 1.0000 - val_loss: 0.0049 - val_precision: 0.9899 - val_recall: 0.9998 - learning_rate: 0.0010
Epoch 4/30
5394/5394 ━━━━━━━━━━━━━━━━━━━━ 337s 62ms/step - accuracy: 0.9982 - auc: 0.9998 - loss: 0.0061 - precision: 0.9980 - recall: 0.9988 - val_accuracy: 0.9964 - val_auc: 1.0000 - va